# Lesson 28: Epipolar Geometry

Lesson 22's stereo matching assumed a *rectified* pair, where corresponding points always fall on the same row. In the more general case, stereo cameras may not be rectified. This lesson covers the general two-view relationship between **any** pair of images of a static scene. As we will see, a point's match in the other image isn't just *somewhere* &mdash; it's constrained to lie on a particular line. We cover both the calibrated and uncalibrated cases.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## The epipolar constraint

For a point $x_1$ in image 1 and its true match $x_2$ in image 2 (both in homogeneous pixel coordinates), there's a $3\times3$ rank-deficient matrix $F$ &mdash; the **fundamental matrix** &mdash; such that

$$x_2^\top F x_1 = 0$$

for *every* corresponding pair, regardless of scene geometry. $F$ depends only on the two cameras' relative pose and (uncalibrated) intrinsics. Rearranged, $l_2 = F x_1$ is the **epipolar line** in image 2: the 1D line along which $x_1$'s match is guaranteed to lie. This is exactly the mechanism that made Lesson 22's row-restricted search valid &mdash; rectification is just the special camera arrangement where every epipolar line happens to be horizontal, which collapses the 1D search to a single row.

### Why a line? A geometric picture

Camera 1 knows only the *direction* to a point, not its depth: the true 3D point $X$ could be anywhere along the ray from $C_1$ through $x_1$. But every point on that ray, together with $C_1$ and $C_2$, lies in a single plane &mdash; the **epipolar plane**. Camera 2 sees this plane edge-on, as a single line: wherever $X$ actually sits along the ray, its projection into image 2 always falls on that one line, the epipolar line.

In [ ]:
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

C1 = np.array([0.0, 0.0, 0.0])
C2 = np.array([3.0, 0.5, -0.5])
ray_dir = np.array([0.6, 0.8, 3.0])
ray_dir = ray_dir / np.linalg.norm(ray_dir)
depths = [2.0, 3.5, 5.0]                        # a few candidate depths along camera 1's ray
candidates = [C1 + d * ray_dir for d in depths]

fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111, projection='3d')

ax.plot(*zip(C1, C2), c='gray', linestyle='--', linewidth=1)
for X in candidates:
    ax.plot(*zip(C1, X), c='tab:blue', linewidth=1)
    ax.plot(*zip(C2, X), c='tab:green', linewidth=1)
cand_arr = np.array(candidates)
ax.scatter(cand_arr[:, 0], cand_arr[:, 1], cand_arr[:, 2], c='black', s=30)
ax.text(*candidates[-1], '  possible $X$\n  (unknown depth)', fontsize=8)

ax.scatter(*C1, c='tab:blue', s=60)
ax.text(*C1, '  $C_1$', fontsize=10)
ax.scatter(*C2, c='tab:green', s=60)
ax.text(*C2, '  $C_2$', fontsize=10)

plane = Poly3DCollection([[C1, C2, candidates[-1]]], alpha=0.15, facecolor='tab:orange')
ax.add_collection3d(plane)
centroid = (C1 + C2 + candidates[1]) / 3
ax.text(*centroid, 'epipolar\nplane', fontsize=8, color='tab:orange', ha='center')

ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
ax.set_title('Why the match is constrained to a line')
plt.tight_layout()
plt.show()

The three blue segments are all rays from $C_1$ through the same pixel $x_1$, just extended to three different, equally plausible depths &mdash; camera 1 alone can't tell them apart. But each candidate, together with $C_1$ and $C_2$, still lies in the one epipolar plane shown, so their projections into camera 2 (the ends of the green segments) all fall along that plane's intersection with image 2: the epipolar line. $F$ is just the algebraic machinery that predicts this line directly from $x_1$, without ever knowing the depth.

## A synthetic two-camera scene

We build two cameras with known intrinsics $K$ and a known relative rotation/translation, create some random 3D points, project the points into both cameras, and use the resulting correspondences to recover $F$.

In [ ]:
rng = np.random.default_rng(0)
K = np.array([[500, 0, 320], [0, 500, 240], [0, 0, 1]], dtype=np.float64)

R1, t1 = np.eye(3), np.zeros(3)                       # camera 1: at the origin, looking down +z
angle = np.radians(15)
R_true = np.array([[np.cos(angle), 0, np.sin(angle)],
                    [0, 1, 0],
                    [-np.sin(angle), 0, np.cos(angle)]])
t_true = np.array([0.5, 0.0, 0.1])                    # camera 2: rotated 15 deg, shifted along x

P1 = K @ np.hstack([R1, t1.reshape(3, 1)])
P2 = K @ np.hstack([R_true, t_true.reshape(3, 1)])

def project(P, points_3d):
    homogeneous = np.hstack([points_3d, np.ones((len(points_3d), 1))])
    projected = (P @ homogeneous.T).T
    return projected[:, :2] / projected[:, 2:3]

points_3d = rng.uniform(-1, 1, (40, 3)) + np.array([0, 0, 5])  # in front of both cameras
x1 = project(P1, points_3d)
x2 = project(P2, points_3d)

print(f'{len(points_3d)} 3D points, projected into both cameras')

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].scatter(x1[:, 0], x1[:, 1], c='tab:blue', s=15)
axes[0].set_xlim(0, 640); axes[0].set_ylim(480, 0)
axes[0].set_title('Image 1')
axes[1].scatter(x2[:, 0], x2[:, 1], c='tab:orange', s=15)
axes[1].set_xlim(0, 640); axes[1].set_ylim(480, 0)
axes[1].set_title('Image 2')
plt.tight_layout()
plt.show()

### SVD: Singular value decomposition

In Lesson 23, we saw that a real symmetric positive-semidefinite matrix $A$ (like a covariance matrix) can be written as $A = P \Lambda P^\top$, with $P$ containing the *eigenvectors*, and with $\Lambda$ diagonal and non-negative containing the *eigenvalues*. The **singular value decomposition (SVD)** generalizes this idea: any matrix $A$ &mdash; symmetric or not, square or not &mdash; can be written as $A = U \Sigma V^\top$, with $U$ and $V$ orthogonal containing the **singular vectors** (left and right, respectively) and $\Sigma$ diagonal and non-negative containing the **singular values**. By definition, the singular values in $\Sigma$ are sorted largest to smallest. For our purposes, we focus on two specific facts about the SVD:

**Solving $Ax=0$ as closely as possible, subject to $\|x\|=1$.** The answer is $V$'s last column (equivalently, $V^\top$'s last row) &mdash; the right singular vector associated with the smallest singular value. This is Lesson 23's total-least-squares trick, generalized: $V$'s columns are the eigenvectors of $A^\top A$, sorted by eigenvalue. So the smallest-singular-value direction here is exactly the same "direction of least fit error" as the smallest-eigenvalue eigenvector Lesson 23 used.

**Enforcing a rank constraint.** Zeroing the smallest singular value(s) and reconstructing gives the closest lower-rank matrix (in a least-squares sense) to the original. That's exactly what's needed to force the estimated $F$ &mdash; which must be exactly rank 2 &mdash; back onto that constraint, after the unconstrained 9-parameter solve inevitably drifts off it slightly.

## The 8-point algorithm

Each correspondence gives one linear equation in the 9 unknown entries of $F$ (expanding $x_2^\top F x_1 = 0$). $F$ is only defined up to scale (like the homography in Lesson 24), so it really has just 8 independent unknowns, solvable as a homogeneous least-squares problem via SVD (above) &mdash; the same DLT machinery used for the homography fit in Lesson 24. Since $F$ must be exactly rank 2 (a fundamental matrix is always singular), we project the 9-parameter solution back onto the nearest rank-2 matrix, using the SVD trick above.

(Note that $F$ actually has only 7 true degrees of freedom, because the rank-2 constraint removes one more; however, any 7-point algorithm will be more complicated because it has to take into account a nonlinear constraint; the 8-point algorithm is nice because it is a *linear* algorithm, even though it requires a separate step to restore the rank-2 property.)

One more wrinkle before we can just hand this to SVD: raw pixel coordinates (in the hundreds) make the linear system badly conditioned numerically. So we first rescale each image's points to be centered at the origin with average distance $\sqrt2$ from it, solve in these normalized coordinates, then undo the rescaling on the resulting $F$. This conditioning step, shown to make a dramatic difference in practice, is what turns the plain 8-point algorithm into the **normalized 8-point algorithm** (<a href="../references.html#hartley-1997">Hartley, 1997</a>) &mdash; the version used below, and the one worth reaching for in practice.

In [ ]:
def normalize_points(x):
    """Shift/scale so points are centered at the origin with average distance sqrt(2) -- standard
    numerical-conditioning trick (Hartley normalization) for the 8-point algorithm."""
    mean = x.mean(axis=0)
    std = x.std()
    T = np.array([[1 / std, 0, -mean[0] / std],
                  [0, 1 / std, -mean[1] / std],
                  [0, 0, 1]])
    x_h = np.hstack([x, np.ones((len(x), 1))])
    return (T @ x_h.T).T, T

def eight_point_algorithm(x1, x2):
    x1n, T1 = normalize_points(x1)
    x2n, T2 = normalize_points(x2)

    A = np.array([[xb * xa, xb * ya, xb, yb * xa, yb * ya, yb, xa, ya, 1]
                  for (xa, ya, _), (xb, yb, _) in zip(x1n, x2n)])
    _, _, Vt = np.linalg.svd(A)
    F = Vt[-1].reshape(3, 3)

    U, S, Vt2 = np.linalg.svd(F)   # enforce rank-2 by zeroing the smallest singular value
    S[-1] = 0
    F = U @ np.diag(S) @ Vt2

    F = T2.T @ F @ T1              # undo the normalization
    return F / F[2, 2]

F_mine = eight_point_algorithm(x1, x2)
F_cv, _ = cv2.findFundamentalMat(x1, x2, cv2.FM_8POINT)

print('our F:\n', np.round(F_mine, 5))
print('cv2.findFundamentalMat F:\n', np.round(F_cv, 5))

### Checking the epipolar constraint directly

For true correspondences, $x_2^\top F x_1$ should be (numerically) zero.

In [ ]:
def epipolar_residual(F, x1, x2):
    x1h = np.hstack([x1, np.ones((len(x1), 1))])
    x2h = np.hstack([x2, np.ones((len(x2), 1))])
    return np.abs(np.sum(x2h * (F @ x1h.T).T, axis=1))

print(f'mean |x2^T F x1|, our F:  {epipolar_residual(F_mine, x1, x2).mean():.2e}')
print(f'mean |x2^T F x1|, cv2 F:  {epipolar_residual(F_cv, x1, x2).mean():.2e}')

### Visualizing epipolar lines

For a handful of points in image 1, we draw their epipolar lines $l_2 = Fx_1$ in image 2, and confirm each true match sits exactly on its line.

In [ ]:
w, h = 640, 480
sample_idx = rng.choice(len(x1), 6, replace=False)
colors = plt.cm.tab10.colors[:len(sample_idx)]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(x1[:, 0], x1[:, 1], c='gray', s=15)
axes[0].scatter(x1[sample_idx, 0], x1[sample_idx, 1], c=colors, s=40)
axes[0].set_xlim(0, w); axes[0].set_ylim(h, 0)
axes[0].set_title('Image 1: 6 selected points')

axes[1].scatter(x2[:, 0], x2[:, 1], c='gray', s=15)
for i, color in zip(sample_idx, colors):
    a, b, c = F_mine @ np.array([x1[i, 0], x1[i, 1], 1])   # line: a*x + b*y + c = 0
    xs = np.array([0, w])
    ys = -(a * xs + c) / b
    axes[1].plot(xs, ys, linewidth=1, color=color)
axes[1].scatter(x2[sample_idx, 0], x2[sample_idx, 1], c=colors, s=40, zorder=5,
                edgecolors='black', linewidths=0.6, label='true match')
axes[1].set_xlim(0, w); axes[1].set_ylim(h, 0)
axes[1].set_title('Image 2: epipolar lines + true matches')
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

Every true match lands exactly on its predicted line &mdash; a point's search in the second image really does collapse from 2D to 1D, even without rectifying the images first.

## From fundamental to essential: adding calibration

The fundamental matrix works in raw pixel coordinates, assuming the camera intrinsics are unknown. If we *do* know the intrinsics $K$ (from camera calibration &mdash; Lesson 27), we can remove them and work in normalized coordinates, giving the **essential matrix**:

$$E = K_2^\top F K_1 \qquad \text{(if both are the same camera, then } K_1 = K_2 = K \text{)}$$

Unlike $F$ which has 7 degrees of freedom, $E$ has only 5 degrees of freedom &mdash; it's built entirely from a relative rotation $R$ and translation direction $t$ between the two cameras: $E = [t]_\times R$, where $[t]_\times$ is the $3\times3$ skew-symmetric "cross-product matrix" of $t$, defined so that $[t]_\times v = t \times v$ for any vector $v$. This means $E$ can be *decomposed* back into $R$ and $t$, which is how you recover camera motion from image correspondences alone.

We estimate $E$ robustly with `cv2.findEssentialMat(..., method=cv2.RANSAC, ...)`, exactly the kind of outlier-rejecting fit built from scratch in Lesson 23.

In [ ]:
E_from_F = K.T @ F_mine @ K
E_direct, _ = cv2.findEssentialMat(x1, x2, K, method=cv2.RANSAC, threshold=1.0)

# E is also only defined up to scale -- compare directions, not raw magnitudes
print('E derived from our F (normalized):\n', np.round(E_from_F / np.linalg.norm(E_from_F), 4))
print('E from cv2.findEssentialMat (normalized):\n', np.round(E_direct / np.linalg.norm(E_direct), 4))

### Recovering camera motion

`cv2.recoverPose` decomposes $E$ into a rotation and a translation *direction* &mdash; not a full translation vector, since the magnitude is fundamentally unrecoverable from two views alone. A scene twice as large, viewed by cameras with twice the baseline between them, produces identical images; this is the same scale ambiguity familiar from monocular vision.

In [ ]:
_, R_estimated, t_estimated, _ = cv2.recoverPose(E_direct, x1, x2, K)

print('true rotation:\n', np.round(R_true, 4))
print('recovered rotation:\n', np.round(R_estimated, 4))
print()
print('true translation direction:      ', np.round(t_true / np.linalg.norm(t_true), 4))
print('recovered translation direction: ', np.round(t_estimated.ravel(), 4))

The rotation is recovered, and the translation direction matches up to the expected sign/scale. From feature correspondences alone (Lesson 20), we've recovered the relative rotation and translation (up to scale) between the two cameras. This is the starting point for structure-from-motion (Lesson 29).

### Exercise

1. Add pixel noise (e.g. std 0.5) to `x1` and `x2` before running the 8-point algorithm. How much does the mean epipolar residual grow, and does normalizing coordinates (as `eight_point_algorithm` does) actually matter here &mdash; try skipping the normalization step and compare.
2. The epipole in image 2 is the projection of camera 1's center, and satisfies $F e_1 = 0$ for the epipole $e_1$ in image 1 (and $F^\top e_2 = 0$ for the epipole $e_2$ in image 2). Compute both epipoles as the null space of $F$ (via SVD) and check whether they fall inside or outside the visible image region for this camera configuration.
3. Increase the rotation angle between the two cameras to 60 degrees and rerun the pose recovery. Does `cv2.recoverPose` still find the correct rotation? At what point would you expect correspondence matching itself (Lesson 20) to become the bottleneck rather than the geometry?